# Abliterate Qwen3-4B-Thinking-2507 with Heretic

Part of the **say-no-more** experiment. This notebook runs
[Heretic](https://github.com/p-e-w/heretic) on the **exact** base model used for
the baseline eval, then uploads the abliterated result to your own Hugging Face
account. Bring that model back to the Mac, convert to 8-bit MLX, and re-run both
refusal suites to compare.

**Base model:** `Qwen/Qwen3-4B-Thinking-2507`  ·  pinned commit `768f209d9ea8…`

### Before you run (Kaggle setup)
1. **Settings → Accelerator → GPU** (T4 or P100; 16 GB is enough for a 4B model).
2. **Settings → Internet → On** (needed for pip, model download, dataset download, HF upload).
3. **Add-ons → Secrets → add `HF_TOKEN`** = a Hugging Face token with **write** access
   (https://huggingface.co/settings/tokens). It is read from the secret store, never typed in.

### What it does
Heretic finds the model's refusal direction from contrasting harmful/harmless prompts,
projects it out of the weights, and runs an Optuna search for ablation parameters that
minimise refusals **and** keep output close to the original (low KL divergence). Heretic is
interactive by design; Kaggle has no terminal, so the driver cell scripts its menu and stops
after a single upload.

## 1 · GPU check + install

> **If a later cell throws a numpy / import error** (e.g. `cannot import name '_slice' from numpy._core.umath`), it's because pip changed numpy under the running kernel. Fix: **Run → Restart & Run All** once. On the fresh kernel the import works.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU — enable it in Settings → Accelerator"
%pip install -q -U heretic-llm

## 2 · Configuration
Edit these if you like. Fewer trials = faster but lower-quality abliteration.

In [ ]:
MODEL_ID     = "Qwen/Qwen3-4B-Thinking-2507"
MODEL_COMMIT = "768f209d9ea81521153ed38c47d515654e938aea"   # exact revision used by the baseline
REPO_SUFFIX  = "Qwen3-4B-Thinking-2507-say-no-more"  # -> <your-username>/<this>
PRIVATE      = False   # True to upload as a private repo
N_TRIALS     = 100     # Heretic default is 200; T4 does ~100 in a few hours
SEED         = 0       # reproducible optimisation

## 3 · Hugging Face login (from the `HF_TOKEN` secret)

In [ ]:
import huggingface_hub
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Could not read the HF_TOKEN secret. Add it under Add-ons -> Secrets "
        "(name: HF_TOKEN, a token with WRITE access), and make sure it is "
        "ATTACHED to this notebook."
    ) from None

huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
USERNAME = huggingface_hub.whoami(token=HF_TOKEN)["name"]
REPO_ID  = f"{USERNAME}/{REPO_SUFFIX}"
print("Will upload to:", REPO_ID, "(private)" if PRIVATE else "(public)")

## 4 · Run Heretic (scripted menu)

The abliteration runs non-interactively. The driver monkeypatches `questionary` so each
menu prompt is answered automatically:

| Heretic prompt | Scripted answer |
|---|---|
| export strategy | **merge** (full merged model — converts to MLX cleanly) |
| which trial to use | **best** (first on the Pareto front) |
| what to do with the model | **upload**, then stop |
| repository name | `REPO_ID` |
| public / private | from `PRIVATE` |
| reproducibility info | **full** (writes `reproduce.json` + model card) |
| HF token | from the secret (no prompt) |

It uploads **once** and then raises a sentinel to break Heretic's menu loop — so the cell
ends cleanly instead of looping forever. Expect this cell to take a few hours on a T4.

In [ ]:
import sys, questionary, huggingface_hub

class _Done(BaseException):
    """Raised to stop Heretic's menu loop after the single upload."""

_state = {"action_calls": 0}

def _cval(choice):
    return getattr(choice, "value", choice)

def _pick(message, choices=None, default=None):
    msg = str(message).lower()
    if "token" in msg:                       # should not appear (we logged in)
        return HF_TOKEN
    if "repository" in msg and "public" not in msg and "private" not in msg:
        return REPO_ID
    if "additional trials" in msg:           # we never request more trials
        return ""
    if "what do you want to do" in msg:      # action menu
        _state["action_calls"] += 1
        if _state["action_calls"] == 1:
            for c in (choices or []):
                if str(_cval(c)).lower() == "upload":
                    return _cval(c)
            return _cval(choices[0]) if choices else "upload"
        raise _Done                          # upload done -> stop the loop
    if choices:
        if "export" in msg:
            for c in choices:
                if str(_cval(c)).lower() == "merge":
                    return _cval(c)
        if "public or private" in msg:
            want = "private" if PRIVATE else "public"
            for c in choices:
                if want in str(_cval(c)).lower():
                    return _cval(c)
        if "reproducibility" in msg:
            for c in choices:
                if "full" in str(_cval(c)).lower():
                    return _cval(c)
        return _cval(choices[0])             # trial -> best; checkpoint -> continue
    return default if default is not None else ""

class _Stub:
    def __init__(self, message, choices=None, default=None, **kw):
        self._m, self._c, self._d = message, choices, default
    def ask(self, *a, **k):        return _pick(self._m, self._c, self._d)
    def unsafe_ask(self, *a, **k): return _pick(self._m, self._c, self._d)

def _factory(*a, **k):
    message = a[0] if a else k.get("message", "")
    return _Stub(message, choices=k.get("choices"), default=k.get("default"))

for _name in ("select", "text", "password", "path", "rawselect", "autocomplete", "checkbox"):
    if hasattr(questionary, _name):
        setattr(questionary, _name, _factory)
questionary.confirm = lambda *a, **k: type("C", (), {
    "ask": lambda s, *x, **y: True, "unsafe_ask": lambda s, *x, **y: True})()

huggingface_hub.get_token = lambda: HF_TOKEN   # Heretic reads the token from here

sys.argv = [
    "heretic",
    "--model", MODEL_ID,
    "--model-commit", MODEL_COMMIT,
    "--export-strategy", "merge",
    "--n-trials", str(N_TRIALS),
    "--seed", str(SEED),
]

from heretic.main import run
try:
    run()
except _Done:
    print("\n✅ Abliteration complete and uploaded to:",
          f"https://huggingface.co/{REPO_ID}")

## 5 · Verify the upload

In [ ]:
from huggingface_hub import list_repo_files
files = list_repo_files(REPO_ID, token=HF_TOKEN)
print("Uploaded files:")
for f in sorted(files):
    print("  ", f)
print("\nModel page: https://huggingface.co/" + REPO_ID)
print("\nNext, on the Mac:")
print(f"  .venv/bin/hf download {REPO_ID} --local-dir models/{REPO_SUFFIX}")
print(f"  .venv/bin/mlx_lm.convert --hf-path models/{REPO_SUFFIX} \\")
print(f"      --mlx-path models/{REPO_SUFFIX}-mlx-8bit -q --q-bits 8")
print("  MODEL=models/%s-mlx-8bit scripts/serve.sh   # then scripts/eval.sh 03-abliterated" % REPO_SUFFIX)